In [ ]:
import ee
import geemap 

Initialize GEE project

In [ ]:
geemap.ee_initialize(project="water-sinapohlabeln") # enter your gee project here

Import modules

In [ ]:
from modules import ms_indices_S2 as indices
from modules import configs, utils_string
from modules import utils_Sentinel2 as utils_LS
from modules import high_level_functions_S2
from modules import utils_geom

Set Parameters

In [ ]:
import numpy as np
# PROPERTIES
# SET METADATA PARAMETERS
MAXCLOUD = 70 #20 
STARTYEAR = 2018
ENDYEAR = 2024
STARTMONTH = 7
ENDMONTH = 8
SCALE = 10

# tile size: 10 degrees longitude x 5 degrees latitude
SIZE_LON = 0.5
SIZE_LAT = 0.5

# processing area: everything from 55 to 85° N
#longitudes = range(-154, -153, SIZE_LON)   #-154, -153, SIZE_LON   ##wainwright: -160.25, -159.75, 70.5, 71,
#latitudes = range(65, 65.5, SIZE_LAT)        # 65, 66, SIZE_LAT, 70.0, 71   

longitudes = np.arange(-163.25, -162.75, SIZE_LON)
latitudes = np.arange(69.5, 70, SIZE_LAT)


BUFFER_SIZE = 0.001

target_collection = 'projects/water-sinapohlabeln/assets/TCSentinel_test'                             #TCTrend_SR_2005-2024_TCVIS' # Target Collection for 2005-2024 data

Image metadata

In [ ]:
# image metadata Filters
config_trend = {
  'STARTYEAR': STARTYEAR,
  'ENDYEAR': ENDYEAR,
  'max_cloud_cover': MAXCLOUD,
  'date_filter_yr' : ee.Filter.calendarRange(STARTYEAR, ENDYEAR, 'year'),
  'date_filter_mth' : ee.Filter.calendarRange(STARTMONTH, ENDMONTH, 'month'),
  'meta_filter_cld' : ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAXCLOUD),
  'select_bands_visible' : ["SR_B1", "SR_B2","SR_B3","SR_B4"],
  'select_indices' : ["TCB", "TCG", "TCW"],
  'select_TCtrend_bands' : ["TCB_slope", "TCG_slope", "TCW_slope"],
  'geom' : None,
  'longitudes' : longitudes,
  'latitudes' : latitudes
}
#------ RUN FULL PROCESS FOR ALL REGIONS IN LOOP ------------------------------

In [ ]:
RUN = 0 # set to 0 or False for debugging mode
m = geemap.Map()

Run & export trends

In [ ]:
for lowLat in latitudes:
    for leftLon in longitudes:
        
        
        # check for Hemisphere
        if lowLat < 0:
            sizeLat = SIZE_LAT * -1
        else:
            sizeLat = SIZE_LAT
            
        sizeLon = SIZE_LON

        # write tile BBox into config
        geom = utils_geom.create_buffered_rectangle(leftLon, lowLat, sizeLon=sizeLon, sizeLat=sizeLat, buffer_size=BUFFER_SIZE, geodesic=False)
        config_trend['geom'] = geom
        # add tile to Map for visualization
        m.addLayer(geom,{}, str(lowLat))

        # Setup automated file name
        assetname_new = 'pointlay_no_ndsi_70'                      #utils_string.make_TCTrendAssetNameSR(leftLon, lowLat, STARTYEAR, ENDYEAR)
        assetId = target_collection + '/' + assetname_new

        # Calculate Trend
        trend = high_level_functions_S2.runTCTrend(config_trend)
        if RUN:
            task = ee.batch.Export.image.toAsset(
                image=ee.Image(trend['visual']).toByte(),
                description=assetname_new,
                assetId=assetId,
                scale=SCALE,
                region=geom,
                maxPixels=1e12)

            task.start()



Testing Sentinel

In [ ]:
# Initialize the map
Map = geemap.Map()

# Run your trend function
trend = high_level_functions_S2.runTCTrend(config_trend)

# Add the visualized trend image, clipped to the geometry
Map.addLayer(
    trend['visual'].clip(config_trend['geom']),
    {
        'min': 0,
        'max': 255,
        'bands': config_trend['select_TCtrend_bands']
    },
    'TC Trend Visual'
)

Map

In [ ]:

## print stats

from modules import utils_Sentinel2
from modules import high_level_functions_S2
import ee
import geemap

# Initialize Earth Engine and geemap
ee.Initialize()
#Map = geemap.Map()
geom = ee.Geometry.Rectangle([-160, 70.65, -159.85, 70.75]) 
config_trend['geom'] = geom
# Run your trend analysis function
results = high_level_functions_S2.runTCTrend(config_trend)

# Extract the visual image
trend_image_visual = results['data']
stats = trend_image_visual.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=config_trend['geom'],
    scale=10,
    maxPixels=1e13
)
print(stats.getInfo())
# Setup visualization parameters (customize this)
#vis_params = {
#    'min': -0.12,
#    'max': 0.,
#    'bands': config_trend['select_TCtrend_bands'],
#}

#Map.addLayer(trend_image_visual, vis_params, 'Trend Visualization')
#Map


In [ ]:
##Dateband testing

from modules import utils_Sentinel2
from modules import high_level_functions_S2
import ee
import geemap

# Initialize Earth Engine and geemap
ee.Initialize()

# Load a Sentinel-2 SR image
image = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2022-06-01', '2022-06-10') \
    .first()
def make_dateband(image):
    ms_to_days = ee.Number(1000 * 60 * 60 * 24)  # 86,400,000
    time = ee.Number(image.get("system:time_start"))
    date_band = ee.Image.constant(time.divide(ms_to_days)).toFloat().rename('Date')
    return image.addBands(date_band)

image_with_date = make_dateband(image)
# Pick a simple geometry for reduction
point = ee.Geometry.Point([13.4050, 52.5200])  # Berlin, example

# Get the value of the Date band at that point
date_value = image_with_date.select('Date').reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=point,
    scale=10
)

print('Date band value:', date_value.getInfo())


In [ ]:
from modules import high_level_functions_S2
trend = high_level_functions_S2.runTCTrend(config_trend)  # Assume this is already defined

for year in range(2017, 2025):
    images = trend['image_collection'] \
        .filterDate(f'{year}-07-01', f'{year}-08-31') \
        .sort('system:time_start') \
        .toList(100)  # max 100 images per year

    count = images.size().getInfo()
    print(f'{year}: {count} images')

In [ ]:
##number of mosaics

from modules import high_level_functions_S2
import ee

# Run your TC trend pipeline
trend = high_level_functions_S2.runTCTrend(config_trend)

# Get the masked original collection and the annual mosaics
masked_collection = trend['image_collection_original']
annual_collection = trend['image_collection']

# Dictionary to store image counts
image_counts = {}

# Loop through each year
for year in range(config_trend['STARTYEAR'], config_trend['ENDYEAR'] + 1):
    # Filter unaggregated images (original, per-pixel masked)
    images = masked_collection.filter(ee.Filter.calendarRange(year, year, 'year'))
    
    # Count the number of images
    count = images.size().getInfo()
    image_counts[year] = count
    print(f'{year}: {count} images')

mosaic_info = annual_collection.filter(ee.Filter.eq('Year', year)).first()

# Check if mosaic exists *safely* by pulling its metadata
if mosaic_info is not None:
    try:
        mosaic_id = mosaic_info.get('name').getInfo()
        print(f'  Mosaic ID: {mosaic_id}')
    except Exception as e:
        print(f'  Could not get mosaic metadata for {year}: {e}')
else:
    print(f'  No mosaic found for {year}')



In [ ]:
##display trend image

from modules import high_level_functions_S2
trend = high_level_functions_S2.runTCTrend(config_trend)

# Create the map
Map = geemap.Map()

# Add the trend visualization layer
Map.addLayer(
    trend['visual'], #.clip(config_trend['geom']),
    {
        'min': 0,
        'max': 255,
    },
    'TC Trend Visual'
)

# Center map and add satellite basemap
Map.centerObject(config_trend['geom'], 8)
Map.add_basemap(basemap='SATELLITE')

# Add the geometry boundary layer
Map.addLayer(config_trend['geom'], {}, str(lowLat))

# Display the map
Map

In [ ]:
import geemap
import ee

# Initialize map
Map = geemap.Map()

# Run trend processing
trend = high_level_functions_S2.runTCTrend(config_trend)
annual_collection = trend['image_collection']

# Convert to list and choose the index (0 = first year)
image_list = annual_collection.toList(annual_collection.size())
index = 2  # Change this to 1, 2, ..., N-1

img = ee.Image(image_list.get(index))

# Visualization
vis_params = {
    'bands': ["SR_B3_median", "SR_B2_median","SR_B1_median"],
    'min': 0,
    'max': 0.2,
}

Map.addLayer(img, vis_params, f'Mosaic index {index}')
Map.addLayerControl()
Map


Visualize Geometry 

In [ ]:
#m